In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time

SEARCH_TERM = "sanctions"
START_DATE = datetime(2022, 1, 3)
END_DATE = datetime(2025, 10, 6)

MAXRECORDS = 250        # max allowed by GDELT for ArtList
RETRY_DELAY = 3         # seconds
MAX_RETRIES = 5

def fetch_day(kw, start, end):
    """
    Fetch one day of articles using GDELT DOC API.
    start/end are datetimes. Returns a list (possibly empty) of article dicts.
    """
    url = "https://api.gdeltproject.org/api/v2/doc/doc"

    params = {
        "query": kw,
        "mode": "ArtList",
        "format": "json",
        "maxrecords": MAXRECORDS,
        "startdatetime": start.strftime("%Y%m%d%H%M%S"),
        "enddatetime": end.strftime("%Y%m%d%H%M%S"),
    }


    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, params=params, timeout=30)

            # Debug if something is wrong
            if r.status_code != 200 or len(r.text.strip()) == 0:
                print(f"⚠️ Empty/invalid response, attempt {attempt+1}/{MAX_RETRIES}")
                time.sleep(RETRY_DELAY)
                continue

            data = r.json()

            if "articles" not in data:
                print(f"⚠️ No articles key, retrying ({attempt+1})")
                time.sleep(RETRY_DELAY)
                continue

            return data["articles"]

        except Exception as e:
            print(f"⚠️ Error: {e} (attempt {attempt+1})")
            time.sleep(RETRY_DELAY)

    print("❌ FAILED after retries — moving on.")
    return []


# ---------------------------
# Build weekly intervals
# ---------------------------

weeks = []
cur = START_DATE
while cur < END_DATE:
    week_end = cur + timedelta(days=6)
    weeks.append((cur, week_end))
    cur = week_end + timedelta(days=1)


# ---------------------------
# Fetch all weeks
# ---------------------------

all_rows = []
weekly_counts = []

for wk_start, wk_end in weeks:
    print(f"📅 Fetching {wk_start.date()} → {wk_end.date()}")

    articles = []
    day = wk_start

    while day <= wk_end:
        next_day = day + timedelta(days=1)

        print(f"   🔎 Daily fetch: {day.date()}")
        day_articles = fetch_day(SEARCH_TERM, day, next_day)

        articles.extend(day_articles)
        day = next_day

    # Count & store weekly totals
    weekly_counts.append({
        "week_start": wk_start.date(),
        "week_end": wk_end.date(),
        "count": len(articles)
    })

    # Add article rows
    for a in articles:
        all_rows.append({
            "week_start": wk_start.date(),
            "week_end": wk_end.date(),
            "url": a.get("url", ""),
            "title": a.get("title", ""),
            "date": a.get("seendate", ""),
            "domain": a.get("domain", ""),
            "language": a.get("language", "")
        })


# ---------------------------
# Save outputs
# ---------------------------

df_articles = pd.DataFrame(all_rows)
df_articles.to_csv("all_articles.csv", index=False)

df_counts = pd.DataFrame(weekly_counts)
df_counts.to_csv("weekly_counts.csv", index=False)

print("\n✅ DONE!")
print(f"Articles saved → all_articles.csv ({len(df_articles)} rows)")
print(f"Weekly counts → weekly_counts.csv ({len(df_counts)} weeks)")

print(f"*DON'T FORGET TO SAVE FILES TO PATH FOR MODEL!!*")

📅 Fetching 2022-01-03 → 2022-01-09
   🔎 Daily fetch: 2022-01-03
   🔎 Daily fetch: 2022-01-04
   🔎 Daily fetch: 2022-01-05
   🔎 Daily fetch: 2022-01-06
   🔎 Daily fetch: 2022-01-07
   🔎 Daily fetch: 2022-01-08
   🔎 Daily fetch: 2022-01-09
📅 Fetching 2022-01-10 → 2022-01-16
   🔎 Daily fetch: 2022-01-10
   🔎 Daily fetch: 2022-01-11
   🔎 Daily fetch: 2022-01-12
   🔎 Daily fetch: 2022-01-13
   🔎 Daily fetch: 2022-01-14
   🔎 Daily fetch: 2022-01-15
   🔎 Daily fetch: 2022-01-16
📅 Fetching 2022-01-17 → 2022-01-23
   🔎 Daily fetch: 2022-01-17
   🔎 Daily fetch: 2022-01-18
   🔎 Daily fetch: 2022-01-19
   🔎 Daily fetch: 2022-01-20
   🔎 Daily fetch: 2022-01-21
   🔎 Daily fetch: 2022-01-22
   🔎 Daily fetch: 2022-01-23
📅 Fetching 2022-01-24 → 2022-01-30
   🔎 Daily fetch: 2022-01-24
   🔎 Daily fetch: 2022-01-25
   🔎 Daily fetch: 2022-01-26
   🔎 Daily fetch: 2022-01-27
   🔎 Daily fetch: 2022-01-28
   🔎 Daily fetch: 2022-01-29
   🔎 Daily fetch: 2022-01-30
📅 Fetching 2022-01-31 → 2022-02-06
   🔎 Daily fe

In [ ]:
import pandas as pd

# Load the CSV already generated
df_articles = pd.read_csv("all_articles.csv")

# Filter for English articles only
df_articles_en = df_articles[df_articles['language'] == 'English']

# Save the filtered dataset
df_articles_en.to_csv("all_articles_en.csv", index=False)

print(f"✅ English-only articles saved → all_articles_en.csv ({len(df_articles_en)} rows)")

✅ English-only articles saved → all_articles_en.csv (56055 rows)


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

# Load English-only articles
df_articles_en = pd.read_csv("all_articles_en.csv")

# Make sure the 'date' column is datetime
df_articles_en['date'] = pd.to_datetime(df_articles_en['date'], errors='coerce')

# Drop any rows where date couldn't be parsed
df_articles_en = df_articles_en.dropna(subset=['date'])

# Set week start date (Monday) for grouping
df_articles_en['week_start'] = df_articles_en['date'].dt.to_period('W-MON').apply(lambda r: r.start_time.date())

# Group by week_start and count articles
weekly_counts_en = df_articles_en.groupby('week_start').size().reset_index(name='count')

# Get week_end for clarity
weekly_counts_en['week_end'] = weekly_counts_en['week_start'] + pd.to_timedelta(6, unit='d')

# Save to CSV
weekly_counts_en.to_csv("weekly_counts_en.csv", index=False)

print(f"✅ Weekly English article counts saved → weekly_counts_en.csv ({len(weekly_counts_en)} weeks)")

/tmp/ipython-input-862152168.py:14: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_articles_en['week_start'] = df_articles_en['date'].dt.to_period('W-MON').apply(lambda r: r.start_time.date())


✅ Weekly English article counts saved → weekly_counts_en.csv (195 weeks)
